<a href="https://colab.research.google.com/github/ali-akbar-bermano/Lecture-series_ALI-AKBAR-BERMANO_G1A024041/blob/main/Implementasi_lecture_series_pertemuan_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Studi Kasus: Analisis Spasial Keterjangkauan Fasilitas Publik & Aset Kota Bengkulu

## 1. Latar Belakang
Kota Bengkulu sebagai pusat pemerintahan dan ekonomi Provinsi Bengkulu terus mengalami perkembangan infrastruktur yang pesat. Dalam perencanaan wilayah yang cerdas (*smart city*), pemetaan visual dan analisis keterjangkauan (*accessibility analysis*) dari aset penting serta fasilitas publik sangat krusial. Pemerintah kota, akademisi, maupun masyarakat memerlukan alat bantu interaktif untuk memahami sebaran geografis fasilitas penting seperti rumah sakit, universitas, cagar budaya, pusat olahraga, hingga gerbang transportasi seperti bandara.

## 2. Rumusan Masalah
* **Sebaran Aset:** Bagaimana sebaran geografis aset-aset penting di Kota Bengkulu secara spasial?
* **Radius Keterjangkauan:** Seberapa luas jangkauan pelayanan (dengan asumsi radius standar 1 kilometer) dari masing-masing fasilitas publik tersebut terhadap area sekitarnya?
* **Visualisasi Interaktif:** Bagaimana menyajikan informasi spasial yang kompleks ini ke dalam peta interaktif berbasis web (*webgis*) yang mudah dipahami oleh pembuat kebijakan maupun masyarakat umum secara gratis dan stabil?

## 3. Solusi & Metodologi
Untuk menjawab tantangan di atas, proyek ini mengimplementasikan analisis spasial otomatis berbasis Python dengan langkah-langkah berikut:
1. **Penyusunan Dataset Spasial:** Mengonversi data tabular koordinat geografis (Lintang dan Bujur) aset Kota Bengkulu menjadi format spasial (*GeoDataFrame*) menggunakan **GeoPandas**.
2. **Analisis Spatial Buffer:** Melakukan proyeksi koordinat dari WGS84 ke sistem proyeksi meteran **UTM Zone 48S (EPSG:32748)** untuk menghitung zona penyangga (*buffer*) dengan radius presisi 1.000 meter (1 km).
3. **Visualisasi WebGIS Interaktif:** Mengintegrasikan peta dengan mesin pembuat peta **Folium**, menggunakan *tiles* dasar dari **Esri World Street Map** untuk menghasilkan peta interaktif yang lengkap dengan penanda (*marker*) berwarna, *popup* informasi detail, dan kontrol lapisan (*layer control*).
```

**Otomatisasi Analisis Spasial & Pemetaan Interaktif (GeoPandas & Folium)**

1. Instalasi Library & Import Module

In [2]:
# 1. Instalasi library geospasial di Google Colab
!pip install geopandas folium shapely pandas matplotlib -q

# 2. Import modul yang dibutuhkan
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.plugins import MarkerCluster

print("✅ Library geospasial berhasil diinstal dan diimpor!")

✅ Library geospasial berhasil diinstal dan diimpor!


2. Pembuatan Dataset Spasial Aset Kota Bengkulu

In [3]:
def load_bengkulu_assets():
    """
    Membuat dataset spasial aset dan fasilitas publik di Kota Bengkulu
    dan mengembalikannya sebagai GeoDataFrame.
    """
    # Sample Dataset Aset Kota Bengkulu
    data_aset = {
        'nama_aset': [
            'RSUD Kota Bengkulu (HD)',
            'Universitas Bengkulu (UNIB Main Campus)',
            'Stadion Semarak Bengkulu',
            'Benteng Marlborough',
            'Bandara Fatmawati Soekarno'
        ],
        'kategori': [
            'Kesehatan',
            'Pendidikan',
            'Olahraga & Publik',
            'Cagar Budaya',
            'Transportasi'
        ],
        'latitude': [-3.8182, -3.7571, -3.8055, -3.7887, -3.8633],
        'longitude': [102.2858, 102.2721, 102.2789, 102.2505, 102.3402]
    }

    # Konversi ke DataFrame
    df = pd.DataFrame(data_aset)

    # Buat geometri Point dari koordinat Longitude dan Latitude (EPSG:4326 / WGS84)
    geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

    return gdf

# Eksekusi fungsi load data
gdf_bengkulu = load_bengkulu_assets()
print(f"✅ Berhasil memuat {len(gdf_bengkulu)} titik aset Kota Bengkulu.")
gdf_bengkulu.head()

✅ Berhasil memuat 5 titik aset Kota Bengkulu.


,nama_aset,kategori,latitude,longitude,geometry
0,RSUD Kota Bengkulu (HD),Kesehatan,-3.8182,102.2858,POINT (102.2858 -3.8182)
1,Universitas Bengkulu (UNIB Main Campus),Pendidikan,-3.7571,102.2721,POINT (102.2721 -3.7571)
2,Stadion Semarak Bengkulu,Olahraga & Publik,-3.8055,102.2789,POINT (102.2789 -3.8055)
3,Benteng Marlborough,Cagar Budaya,-3.7887,102.2505,POINT (102.2505 -3.7887)
4,Bandara Fatmawati Soekarno,Transportasi,-3.8633,102.3402,POINT (102.3402 -3.8633)


3. Analisis Spasial Keterjangkauan (Spatial Buffer Analysis)

In [4]:
def calculate_spatial_buffers(gdf, radius_meters=1000):
    """
    Menghitung zona buffer (radius keterjangkauan dalam meter) untuk setiap titik aset.
    Menggunakan re-proyeksi ke UTM Zone 48S (EPSG:32748) khusus wilayah Bengkulu/Sumatera.
    """
    # Re-proyeksi dari WGS84 (derajat) ke UTM Zone 48S (meter)
    gdf_utm = gdf.to_crs(epsg=32748)

    # Hitung buffer sesuai jarak meter
    gdf_utm['buffer_geom'] = gdf_utm.geometry.buffer(radius_meters)

    # Buat GeoDataFrame khusus untuk polygon buffer dan kembalikan ke sistem WGS84 (EPSG:4326)
    gdf_buffer = gpd.GeoDataFrame(gdf_utm, geometry='buffer_geom', crs="EPSG:32748")
    gdf_buffer = gdf_buffer.to_crs(epsg=4326)

    return gdf_buffer

# Hitung buffer radius 1.000 meter (1 km)
radius_km = 1.0  # 1 km
gdf_buffered = calculate_spatial_buffers(gdf_bengkulu, radius_meters=radius_km * 1000)
print(f"✅ Analisis Spatial Buffer {radius_km} km berhasil dihitung!")

✅ Analisis Spatial Buffer 1.0 km berhasil dihitung!


4. Pembuat Peta Web Interaktif (Folium Web Mapping Engine)

In [5]:
def create_interactive_map(gdf_points, gdf_buffers):
    """
    Menghasilkan Peta Interaktif Folium dengan Marker Aset dan Area Buffer.
    Menggunakan penyedia peta Esri World Street Map yang stabil & gratis.
    """
    # Titik tengah Kota Bengkulu untuk inisialisasi peta
    bengkulu_center = [-3.8000, 102.2600]

    # 💡 SOLUSI: Gunakan Esri World Street Map (Gratis, Stabil, tanpa API Key)
    m = folium.Map(
        location=bengkulu_center,
        zoom_start=12,
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
        attr='Tiles &copy; Esri &mdash; Source: Esri, DeLorme, NAVTEQ, USGS, Intermap, iPC, NRCAN, Esri Japan, METI, Esri China (Hong Kong), Esri (Thailand), TomTom, 2012'
    )

    # Pemetaan Warna berdasarkan Kategori
    color_dict = {
        'Kesehatan': 'red',
        'Pendidikan': 'blue',
        'Olahraga & Publik': 'green',
        'Cagar Budaya': 'orange',
        'Transportasi': 'purple'
    }

    # 1. Tambahkan Lapisan Polygon Buffer (Keterjangkauan Radius 1 km)
    for _, row in gdf_buffers.iterrows():
        folium.GeoJson(
            row['buffer_geom'],
            style_function=lambda x, color=color_dict.get(row['kategori'], 'gray'): {
                'fillColor': color,
                'color': color,
                'weight': 1,
                'fillOpacity': 0.2
            },
            tooltip=f"Zona Radius 1 km: {row['nama_aset']}"
        ).add_to(m)

    # 2. Tambahkan Marker Titik Aset
    for _, row in gdf_points.iterrows():
        icon_color = color_dict.get(row['kategori'], 'gray')

        # Konten Popup HTML
        popup_html = f"""
        <div style="font-family: Arial; width: 200px;">
            <h4><b>{row['nama_aset']}</b></h4>
            <hr style="margin: 5px 0;">
            <b>Kategori:</b> {row['kategori']}<br>
            <b>Koordinat:</b> {row['latitude']:.4f}, {row['longitude']:.4f}
        </div>
        """

        folium.Marker(
            location=[row['latitude'], row['longitude']],
            popup=folium.Popup(popup_html, max_width=250),
            tooltip=row['nama_aset'],
            icon=folium.Icon(color=icon_color, icon='info-sign')
        ).add_to(m)

    # Tambahkan Layer Control
    folium.LayerControl().add_to(m)

    return m

print("✅ Fungsi pembuat peta interaktif berhasil diperbarui dengan provider Esri.")

✅ Fungsi pembuat peta interaktif berhasil diperbarui dengan provider Esri.


5. Eksekusi & Menampilkan Peta Interaktif

In [6]:
# Jalankan dan Tampilkan Peta Interaktif
peta_bengkulu = create_interactive_map(gdf_bengkulu, gdf_buffered)

# Tampilkan Peta di Colab
peta_bengkulu